# Kaggle — entrenamiento (F2.8)

Plantilla derivada de `kaggle_setup.ipynb`. Antes de correr:

1. **Add Data → Your Datasets → `melanoma-isic2020-splits`** (manifiesto, splits, `SHA256SUMS`).
2. **Add Data → Your Datasets → `melanoma-isic2020-512`** (imágenes redimensionadas, F2.0).
3. **Accelerator → GPU**, Internet activado.
4. **Add-ons → Secrets → `WANDB_API_KEY`** (la llave nunca va en el notebook ni en el repositorio).

El código que corre es literalmente el del repositorio en el commit `REPO_SHA`: una sola fuente de
verdad, y la corrida queda atada a ese commit (W&B registra el SHA y el hash de los splits).
Solo se leen `train.txt` y `val.txt`. Verificado en local con `make check-notebooks`.


In [ ]:
# Parámetros de la corrida
REPO_URL = "https://github.com/Edgar-Ontiveros/melanoma-triage.git"
REPO_SHA = "main"  # ← SHA del commit a ejecutar, nunca `main` en una corrida real
# Overrides de Hydra. Ejemplos:
#   B1 con pos_weight, semilla 0:   ["+experiment=b1_resnet50_224", "train.seed=0"]
#   B1 con muestreo ponderado:      ["+experiment=b1_resnet50_224_sampler", "train.seed=0"]
#   F2.6 condición B:               ["+experiment=prep_b_divide255"]
OVERRIDES = ["+experiment=b1_resnet50_224", "train.seed=0"]
RUN_TAG = "b1-s0"
SPLITS_DIR = "/kaggle/input/datasets/edgaronti26/melanoma-isic2020-splits"
# None → data.paths.kaggle.images_dir de configs/data/default.yaml del commit clonado (una sola
# fuente de verdad; el 2026-09-17 una copia hardcodeada aquí apuntaba a la carpeta equivocada).
IMAGES_DIR = None
WORK = "/kaggle/working"
NUM_WORKERS = 4
# /kaggle/input es un montaje remoto; 15 épocas leen 23 mil archivos cada una. Copiar las
# imágenes (1.56 GB) al disco local del kernel una vez cuesta unos minutos y quita esa latencia.
COPY_IMAGES_LOCAL = True

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_DIR = f"{WORK}/repo"
subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REPO_SHA], check=True)
os.chdir(REPO_DIR)
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("commit", SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[train]"], check=True)
# El .pth de la instalación editable solo se lee al arrancar el intérprete: este kernel ya está
# corriendo y no vería src/. Sin esto, `import melanoma` resolvía al directorio del clon como
# paquete namespace y `melanoma.data` era la carpeta data/ del repo (F2, 2026-09-17).
sys.path.insert(0, f"{REPO_DIR}/src")
importlib.invalidate_caches()

In [ ]:
from pathlib import Path

from omegaconf import OmegaConf

# Verificar los hashes de los splits contra SHA256SUMS. Falla si no coinciden.
from melanoma.data.datamodule import verify_split_hashes

if IMAGES_DIR is None:
    IMAGES_DIR = str(OmegaConf.load("configs/data/default.yaml").paths.kaggle.images_dir)
print("imágenes:", IMAGES_DIR)
hashes = verify_split_hashes(SPLITS_DIR)
print("splits íntegros:", hashes)
assert Path(IMAGES_DIR).exists(), f"no está adjunto el dataset de imágenes: {IMAGES_DIR}"
n_src = sum(1 for _ in Path(IMAGES_DIR).glob("*.jpg"))
assert n_src > 0, f"no hay JPEG directamente en {IMAGES_DIR}: revisar data.paths.kaggle.images_dir"
if COPY_IMAGES_LOCAL:
    import shutil
    import time

    t0 = time.time()
    local_images = Path(WORK) / "images_512"
    if not local_images.exists():
        shutil.copytree(IMAGES_DIR, local_images)
    IMAGES_DIR = str(local_images)
    print(f"imágenes copiadas al disco local en {(time.time() - t0) / 60:.1f} min")
n_jpg = sum(1 for _ in Path(IMAGES_DIR).glob("*.jpg"))
print(f"{n_jpg} imágenes en {IMAGES_DIR}")
assert n_jpg == n_src, f"la copia local tiene {n_jpg} JPEG y el origen {n_src}"

In [ ]:
# Llave de W&B desde Kaggle Secrets. Si no existe, la corrida sigue en modo offline.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: llave cargada desde Secrets")
except Exception as exc:
    print("W&B sin llave (modo offline):", exc)

In [ ]:
# Entrenar con configs/ del repositorio, en un intérprete nuevo. Las rutas de Kaggle se pasan
# explícitas (data.env=kaggle) para no depender de la detección automática.
run_dir = f"{WORK}/runs/{RUN_TAG}-{SHA}"
cmd = [
    sys.executable,
    "scripts/train.py",
    *OVERRIDES,
    "data.env=kaggle",
    f"data.paths.kaggle.images_dir={IMAGES_DIR}",
    f"data.paths.kaggle.splits_dir={SPLITS_DIR}",
    f"data.paths.kaggle.manifest_path={SPLITS_DIR}/isic2020.csv",
    f"data.num_workers={NUM_WORKERS}",
    f"hydra.run.dir={run_dir}",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Artefactos en WORK: checkpoint, metrics.json, curvas, figuras y resumen.
import json

summary = json.loads(Path(run_dir, "metrics.json").read_text())
keys = ("run_name", "best_checkpoint", "epochs_run", "collapse_epochs")
print(json.dumps({k: summary[k] for k in keys}, indent=2))
print(json.dumps(summary["metrics"], indent=2))
print(Path(run_dir, "summary.md").read_text())

## Después de la corrida

Descargar `metrics.json`, `config.yaml`, `val_predictions.csv`, `summary.md` y `figures/` del
directorio de la corrida (`/kaggle/working/runs/...`) y copiarlos a `reports/runs/<run_name>/` en el
repositorio; `make f2-report` arma `reports/f2_baselines.md` y `reports/preprocessing_experiment.md`.
El checkpoint (`.ckpt`) no se versiona.
